In [1]:
from prefect_aws.s3 import S3Bucket
from prefect import flow, task
import pandas as pd
from io import BytesIO

@task(name = 'Connect S3')
def connect_s3(bucket = "portfolio-project-files"):
    s3 = S3Bucket.load(bucket)
    return s3

@task(name = 'Download S3 file')
def download_s3(s3, path):
    bytes = s3.read_path(path)
    data = pd.read_csv(BytesIO(bytes))
    return data

@task(name = 'Upload S3 file')
def upload_s3(s3, path, data):
    s3.write_path(path, bytes(data.to_csv(index=False), encoding='utf-8'))

@flow(name = 'test')
def master():
    s3 = connect_s3()
    data = download_s3(s3, 'sdg-bill-tracking/sdg_indicators_corpus.csv')
    return data


data = master()


10:19:32.911 | INFO    | Flow run 'mellow-sawfish' - Beginning flow run 'mellow-sawfish' for flow 'test'

10:19:32.922 | INFO    | Flow run 'mellow-sawfish' - View at https://app.prefect.cloud/account/3f9bd1dc-a34b-4ba7-a6e0-e2aa163f25d6/workspace/4fe3e874-7162-4c73-879d-d1f78dbe5925/runs/flow-run/06993358-4941-7170-8000-7e98f078799d

10:19:33.276 | INFO    | Task run 'Connect S3-0ee' - Finished in state Completed()

10:19:34.374 | INFO    | Task run 'Download S3 file-502' - Finished in state Completed()

10:19:34.567 | INFO    | Flow run 'mellow-sawfish' - Finished in state Completed()

In [2]:
data = data.groupby(['SDG No.', 'Target No.', 'SDG', 'Target'])['Indicator'].apply(lambda x: '\n'.join(x)).reset_index().sort_values(by = ['SDG No.', 'SDG']).reset_index(drop = True).rename(columns = {'Indicator' : 'Indicators'})


In [3]:
data

,SDG No.,Target No.,SDG,Target,Indicators
0,1,1.1,Goal 1. End poverty in all its forms everywhere,"1.1 By 2030, eradicate extreme poverty for all...",1.1.1 Proportion of the population living belo...
1,1,1.2,Goal 1. End poverty in all its forms everywhere,"1.2 By 2030, reduce at least by half the propo...",1.2.1 Proportion of population living below th...
2,1,1.3,Goal 1. End poverty in all its forms everywhere,1.3 Implement nationally appropriate social pr...,1.3.1 Proportion of population covered by soci...
3,1,1.4,Goal 1. End poverty in all its forms everywhere,"1.4 By 2030, ensure that all men and women, in...",1.4.1 Proportion of population living in house...
4,1,1.5,Goal 1. End poverty in all its forms everywhere,"1.5 By 2030, build the resilience of the poor ...","1.5.1 Number of deaths, missing persons and di..."
...,...,...,...,...,...
163,17,17.5,Goal 17. Strengthen the means of implementatio...,17.5 Adopt and implement investment promotion ...,17.5.1 Number of countries that adopt and impl...
164,17,17.6,Goal 17. Strengthen the means of implementatio...,"17.6 Enhance North-South, South-South and tria...",17.6.1 Fixed broadband subscriptions per 100 i...
165,17,17.7,Goal 17. Strengthen the means of implementatio...,"17.7 Promote the development, transfer, dissem...",17.7.1 Total amount of funding for developing ...
166,17,17.8,Goal 17. Strengthen the means of implementatio...,17.8 Fully operationalize the technology bank ...,17.8.1 Proportion of individuals using the Int...
